In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

True Tesla T4


In [2]:
!git clone https://github.com/juuanmartiinez/offside-detector.git
import sys; sys.path.append("/content/offside-detector")

from google.colab import drive
drive.mount('/content/drive')

Cloning into 'offside-detector'...
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 19 (delta 0), reused 19 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 9.11 MiB | 14.74 MiB/s, done.
Mounted at /content/drive


In [6]:
import os
BASE = "/content/drive/MyDrive/Colab Notebooks/football-players"
PESOS = "/content/drive/MyDrive/Colab Notebooks/modelo_16ep.pt"
print(os.listdir(BASE))
print(len(os.listdir(BASE + "/train")))

['README.roboflow.txt', 'README.dataset.txt', 'test', 'valid', 'train']
299


In [45]:
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader
from src.CocoDataset import CocoDataset
from src.DetectionDataset import DetectionDataset, collate
from src.modelo import crearModelo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("entrenando en:", device)

ds = CocoDataset(BASE + "/train")
loader = DataLoader(DetectionDataset(ds), batch_size=4, shuffle=True, collate_fn=collate)

modelo = crearModelo(ligero=False, congelarBackbone=False)
#modelo.load_state_dict(torch.load(PESOS, map_location=device))
modelo.to(device)

optimizer = torch.optim.SGD(
    [p for p in modelo.parameters() if p.requires_grad],
    lr=0.0005, momentum=0.9
)

entrenando en: cuda
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 168MB/s]


In [46]:
modelo.train()

epocas = 10

for epoca in range(epocas):
    total = 0.0
    for n, (img, target) in enumerate(loader, 1):
        imagenes = [i.to(device) for i in img]
        targets = [{"boxes": t["boxes"].to(device),
                    "labels": t["labels"].to(device)} for t in target]

        perdidas = modelo(imagenes, targets)
        loss = sum(perdidas.values())
        total += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"=== epoca {epoca}: coste medio {total/n:.4f} ===")



=== epoca 0: coste medio 1.3648 ===
=== epoca 1: coste medio 0.9624 ===
=== epoca 2: coste medio 0.8749 ===
=== epoca 3: coste medio 0.8200 ===
=== epoca 4: coste medio 0.7804 ===
=== epoca 5: coste medio 0.7514 ===
=== epoca 6: coste medio 0.7225 ===
=== epoca 7: coste medio 0.6970 ===
=== epoca 8: coste medio 0.6759 ===
=== epoca 9: coste medio 0.6605 ===


In [49]:
torch.save(modelo.state_dict(), "/content/drive/MyDrive/Colab Notebooks/modelo_experimento_ResNet.pt")

In [48]:
from torchvision.transforms.functional import to_tensor
from src.metricas import evaluar

dsTest = CocoDataset(BASE + "/valid")

modelo.eval()

PERSONAS = {2, 3, 4}
UMBRAL = 0.2

preds = {}
for i in sorted(dsTest.id2fichero):
    x = to_tensor(dsTest.imagen(i)).to(device)

    with torch.no_grad():
        p = modelo([x])[0]

    cajas = [
        b for b, l, s in zip(p["boxes"].tolist(), p["labels"].tolist(), p["scores"].tolist())
        if int(l) in PERSONAS and s >= UMBRAL
    ]

    preds[i] = {"boxes" : cajas}

recall, precision = evaluar(dsTest, preds)
print(f"recall {recall:.3f}  precision {precision:.3f}")


recall 0.948  precision 0.732
